# RobustRAG Table I reproduction

This notebook reproduces the Mistral-7B keyword-aggregation results from Table I of *Certifiably Robust RAG against Retrieval Corruption*. It keeps the two paper versions separate:

- **Legacy/code profile:** `alpha=0.3`, `beta=3`
- **Current attached-paper profile:** `alpha=0.2`, `beta=3`

Use an **A100 or L4** Colab runtime and run cells in order. The paper used an A100; L4 results are recorded as a hardware-deviated reproduction. The notebook creates an isolated Python 3.11 environment for the authors' 2024 dependencies, downloads the exact upstream commit into ephemeral `/content`, verifies dataset checksums, reuses cached LLM responses, and saves every experiment in a new directory under the shared Drive.

The upstream repository has no license file, so its source is not copied into this project. It is fetched directly from the authors and pinned to commit `9bc35b2fa5fa7d1088383b2789ec19a512d316b1`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
PROJ = "/content/drive/MyDrive/colab/adversarial_rag"
SHARED = "/content/drive/MyDrive/colab/adversarial_rag-shared"
os.chdir(PROJ)
sys.path.insert(0, PROJ)

def run_checked(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

# Colab's Python 3.13 cannot install the authors' 2024 package versions.
# Keep the notebook kernel unchanged and run the experiment in Python 3.11.
run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = Path("/content/robustrag-py311")
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])
SPACY_MODEL = ("https://github.com/explosion/spacy-models/releases/download/"
               "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl")
run_checked([*UV, "pip", "install", "--python", PYTHON,
             "torch==2.2.1", "numpy==1.26.4",
             "transformers==4.40.1", "tokenizers==0.19.1",
             "accelerate==0.29.3", "huggingface-hub==0.23.5",
             "sentencepiece==0.2.0", "joblib==1.3.2",
             "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
             "setuptools<81", SPACY_MODEL])
RUN_ENV = dict(os.environ, NLTK_DATA="/content/robustrag-nltk-data",
               PYTHONHASHSEED="0")
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]
run_checked([PYTHON, "-m", "nltk.downloader", "-d", RUN_ENV["NLTK_DATA"],
             "stopwords"], env=RUN_ENV)
print("Experiment Python:", PYTHON)

## Personal output folder and hardware check

Enter your name when prompted. The notebook is shared, but experiment outputs remain separated as required by `ONBOARDING.md`.

In [ ]:
from pathlib import Path
import subprocess

OWNER = input("Output folder name (for example, manya or connor): ").strip()
if not OWNER or OWNER.lower() in {"replace-me", "your-name"}:
    raise ValueError("Enter a non-empty personal output folder name.")

DATA_DIR = Path(SHARED) / "data" / "robustrag"
RUNS_DIR = Path(SHARED) / "outputs" / OWNER / "robustrag-reproduction"
UPSTREAM_DIR = Path(PROJ) / "third_party" / "robustrag_upstream"
RUNNER_DIR = Path("/content/robustrag-runner")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu)
supported_gpus = ("A100", "L4")
if not any(gpu_name in gpu for gpu_name in supported_gpus):
    raise RuntimeError("Select an A100 or L4 GPU runtime.")
if "L4" in gpu:
    print("Note: the paper used an A100; this run will be labeled as an L4 reproduction.")
print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)

## Prepare the vendored authors' code and verified data

The pinned upstream Python source is committed under `third_party/`, so this step makes no GitHub code download. Shared JSON datasets stay outside git and are downloaded only when missing, with SHA-256 verification.

In [ ]:
from reproduction.robustrag_setup import (
    DATASETS, UPSTREAM_COMMIT, build_runner, ensure_datasets,
    file_sha256, verify_vendored_source,
)

build_runner(UPSTREAM_DIR, RUNNER_DIR)
ensure_datasets(DATA_DIR)

preflight = """
import json
import spacy, torch, transformers
from nltk.corpus import stopwords
from src.defense import KeywordAgg
from src.models import create_model
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'
assert transformers.__version__ == '4.40.1'
assert 'the' in stopwords.words('english')
assert spacy.load('en_core_web_sm')('Paris')[0].pos_
print(json.dumps({'python': __import__('sys').version,
                  'torch': torch.__version__,
                  'transformers': transformers.__version__,
                  'gpu': torch.cuda.get_device_name(0)}))
"""
preflight_result = subprocess.run(
    [PYTHON, "-c", preflight], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, capture_output=True, text=True,
)
print("Preflight:", preflight_result.stdout.strip())
subprocess.run(
    [PYTHON, RUNNER_DIR / "main.py", "--help"], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, stdout=subprocess.DEVNULL,
)
print("Upstream entry point imports successfully.")

## Reproduction runner

The functions below run the authors' `main.py` with the keyword/certification algorithms unchanged. The only source compatibility change defers imports used exclusively by the unused GPT path. Runs use temporary local storage, then copy logs, responses, and caches into a new timestamped shared-output directory. Earlier runs and caches are never modified.

In [ ]:
import json
import re
import shutil
import tempfile
from datetime import datetime, timezone

PROFILES = {
    "legacy_v1": {
        "alpha": 0.3, "beta": 3.0,
        "expected": {
            "realtimeqa": {"no_rag": 8, "vanilla": 69, "keyword": 59, "certified": 45},
            "open_nq": {"no_rag": 30, "vanilla": 61, "keyword": 54, "certified": 47},
        },
    },
    "current_v2": {
        "alpha": 0.2, "beta": 3.0,
        "expected": {
            "realtimeqa": {"no_rag": 8, "vanilla": 69, "keyword": 71, "certified": 38},
            "open_nq": {"no_rag": 30, "vanilla": 61, "keyword": 61, "certified": 26},
        },
    },
}

def package_versions():
    script = """
import importlib.metadata, json
names = ['accelerate', 'en-core-web-sm', 'huggingface-hub', 'joblib', 'nltk',
         'numpy', 'sentencepiece', 'spacy', 'tokenizers', 'torch', 'transformers']
print(json.dumps({name: importlib.metadata.version(name) for name in names}))
"""
    output = subprocess.check_output(
        [PYTHON, "-c", script], env=RUN_ENV, text=True,
    )
    return json.loads(output)

def upstream_command(condition, dataset, profile):
    common = [str(PYTHON), str(RUNNER_DIR / "main.py"), "--model_name", "mistral7b", "--dataset_name", dataset, "--save_response", "--use_cache"]
    if condition == "no_rag":
        return common + ["--top_k", "0", "--defense_method", "none", "--corruption_size", "0"]
    if condition == "vanilla":
        return common + ["--top_k", "10", "--defense_method", "none", "--corruption_size", "0"]
    if condition == "keyword":
        return common + ["--top_k", "10", "--defense_method", "keyword", "--corruption_size", "1", "--alpha", str(profile["alpha"]), "--beta", str(profile["beta"]), "--no_vanilla"]
    raise ValueError(f"Unknown condition: {condition}")

def stream_command(command, cwd, transcript):
    print("Running:", " ".join(command), flush=True)
    with Path(transcript).open("w") as output:
        process = subprocess.Popen(command, cwd=cwd, env=RUN_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            output.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Upstream command failed with exit code {return_code}")

def parse_log(log_path, condition, sample_size):
    text = Path(log_path).read_text()
    def count(label):
        matches = re.findall(rf"{re.escape(label)}:\s*(\d+)", text)
        return int(matches[-1]) if matches else None
    counts = {
        "undefended_correct": count("undefended_corr_cnt"),
        "defended_correct": count("defended_corr_cnt"),
        "certified_correct": count("certify_cnt"),
    }
    primary = counts["undefended_correct"] if condition in {"no_rag", "vanilla"} else counts["defended_correct"]
    if primary is None:
        raise RuntimeError(f"Could not parse metrics from {log_path}")
    metrics = {"sample_size": sample_size, "correct_count": primary, "accuracy": primary / sample_size, "counts": counts}
    if condition == "keyword":
        metrics["certified_accuracy"] = counts["certified_correct"] / sample_size
    return metrics

def copy_runtime_artifacts(runtime, run_dir):
    for name in ["cache", "log", "result", "result_certify"]:
        source = runtime / name
        if source.exists():
            shutil.copytree(source, run_dir / name, dirs_exist_ok=False)

def run_table1(profile_name, dataset, conditions=("keyword",), limit=100, cache_from=None):
    if profile_name not in PROFILES or dataset not in DATASETS:
        raise ValueError("Unknown profile or dataset.")
    if not 1 <= limit <= 100:
        raise ValueError("limit must be between 1 and 100.")
    if len(set(conditions)) != len(conditions):
        raise ValueError("Conditions must be unique.")
    verify_vendored_source(UPSTREAM_DIR)
    if not (RUNNER_DIR / "main.py").is_file():
        raise RuntimeError("The prepared Python 3.11 runner is missing; rerun setup.")

    profile = PROFILES[profile_name]
    filename, expected_hash = DATASETS[dataset]
    source_dataset = DATA_DIR / filename
    if file_sha256(source_dataset) != expected_hash:
        raise RuntimeError(f"Dataset checksum mismatch: {source_dataset}")

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = RUNS_DIR / f"table1-{profile_name}-{dataset}-{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=False)
    manifest = {
        "status": "running", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "profile": profile_name, "dataset": dataset, "dataset_sha256": expected_hash,
        "upstream_commit": UPSTREAM_COMMIT, "compatibility_patch": "defer GPT-only litellm/openai imports; Mistral, keyword aggregation, and certification code unchanged", "sample_limit": limit, "conditions": list(conditions),
        "runner_sha256": {"main.py": file_sha256(RUNNER_DIR / "main.py"), "src/models.py": file_sha256(RUNNER_DIR / "src" / "models.py"), "src/defense.py": file_sha256(RUNNER_DIR / "src" / "defense.py")},
        "parameters": {"model": "mistralai/Mistral-7B-Instruct-v0.2", "top_k": 10, "group_size": 1, "corruption_size": 1, "alpha": profile["alpha"], "beta": profile["beta"], "greedy_decoding": True},
        "expected_full_sample_counts": profile["expected"][dataset],
        "environment": {"python": json.loads(preflight_result.stdout)["python"], "gpu": gpu, "hardware_matches_paper": "A100" in gpu, "packages": package_versions()},
        "commands": [],
    }
    manifest_path = run_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))

    try:
        with tempfile.TemporaryDirectory(prefix="robustrag-run-", dir="/content") as temporary:
            runtime = Path(temporary)
            records = json.loads(source_dataset.read_text())[:limit]
            (runtime / "data").mkdir()
            (runtime / "data" / filename).write_text(json.dumps(records))
            if cache_from is not None:
                cache_source = Path(cache_from)
                if (cache_source / "cache").is_dir():
                    cache_source = cache_source / "cache"
                shutil.copytree(cache_source, runtime / "cache")
            metrics = {}
            try:
                for condition in conditions:
                    old_logs = set((runtime / "log").glob("*.log"))
                    command = upstream_command(condition, dataset, profile)
                    manifest["commands"].append(command)
                    manifest_path.write_text(json.dumps(manifest, indent=2))
                    stream_command(command, runtime, run_dir / f"{condition}.txt")
                    new_logs = set((runtime / "log").glob("*.log")) - old_logs
                    if not new_logs:
                        raise RuntimeError("The upstream run did not create a new log.")
                    log_path = max(new_logs, key=lambda path: path.stat().st_mtime_ns)
                    condition_metrics = parse_log(log_path, condition, len(records))
                    if len(records) == 100:
                        expected = profile["expected"][dataset]
                        condition_metrics["expected_correct_count"] = expected[condition]
                        condition_metrics["matches_expected_correct_count"] = condition_metrics["correct_count"] == expected[condition]
                        if condition == "keyword":
                            condition_metrics["expected_certified_count"] = expected["certified"]
                            condition_metrics["matches_expected_certified_count"] = condition_metrics["counts"]["certified_correct"] == expected["certified"]
                    metrics[condition] = condition_metrics
                    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
            finally:
                copy_runtime_artifacts(runtime, run_dir)
        manifest["status"] = "complete"
        manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        manifest_path.write_text(json.dumps(manifest, indent=2))
        print("Completed:", run_dir)
        print(json.dumps(metrics, indent=2))
        return run_dir
    except Exception:
        manifest["status"] = "failed"
        manifest["failed_at_utc"] = datetime.now(timezone.utc).isoformat()
        manifest_path.write_text(json.dumps(manifest, indent=2))
        raise

## 1. Two-question smoke test

This downloads Mistral-7B and verifies that the legacy pipeline completes before spending time on the full sample.

In [ ]:
smoke_run = run_table1(
    profile_name="legacy_v1",
    dataset="realtimeqa",
    conditions=("keyword",),
    limit=2,
)

## 2. Full RealtimeQA reproduction

Run the legacy/code profile first. The current-paper run then copies its cache, so isolated responses are not regenerated.

In [ ]:
legacy_rqa_run = run_table1(
    profile_name="legacy_v1",
    dataset="realtimeqa",
    conditions=("no_rag", "vanilla", "keyword"),
)

In [ ]:
current_rqa_run = run_table1(
    profile_name="current_v2",
    dataset="realtimeqa",
    conditions=("no_rag", "vanilla", "keyword"),
    cache_from=legacy_rqa_run,
)

## 3. Full Natural Questions reproduction

Again, run the legacy profile first and reuse its cache for the current-paper profile. The upstream file contains 500 questions, but Table I and this notebook use its first 100.

In [ ]:
legacy_nq_run = run_table1(
    profile_name="legacy_v1",
    dataset="open_nq",
    conditions=("no_rag", "vanilla", "keyword"),
)

In [ ]:
current_nq_run = run_table1(
    profile_name="current_v2",
    dataset="open_nq",
    conditions=("no_rag", "vanilla", "keyword"),
    cache_from=legacy_nq_run,
)

## 4. Compare observed and expected counts

An exact match is ideal because these are the same fixed 100 questions. If only the current profile differs, report the repository/paper revision gap rather than tuning parameters until it matches.

In [ ]:
completed_runs = {
    "legacy_rqa": legacy_rqa_run,
    "current_rqa": current_rqa_run,
    "legacy_nq": legacy_nq_run,
    "current_nq": current_nq_run,
}
for label, run_path in completed_runs.items():
    print(f"\n{label}: {run_path}")
    print((Path(run_path) / "metrics.json").read_text())

## 5. Promotion/suppression feasibility profiler

This is deliberately a **response-level oracle analysis**, not yet a claim that a natural-looking injected passage can force the desired isolated response.

- **Promotion:** choose a non-gold phrase that the defense actually extracted from a benign response but currently filtered out. Add one vote per injected passage and test whether it crosses the threshold.
- **Suppression:** add distinct semantic non-answers such as 'Unknown marker 1'. RobustRAG counts them as answers because its abstention check only recognizes the literal substring 'I don't'. Because top-k is fixed, this raises the threshold only when the injected passage displaces an abstaining benign passage.
- **Vote loss:** separately record cases where injecting passages removes low-ranked correct support even before the answer-count threshold rises.

The profiler reports changes to the retained keyword set. Gold-aligned phrases are identified with a documented string-matching heuristic and must be manually audited. Structural keyword changes are neither sufficient nor necessary for the final LLM answer to become wrong, so live passage construction and end-to-end validation remain a separate stage. In that stage, first use claim-copying or genuinely claim-free passages; treat any passage-level instruction that forces a chosen response as a separate prompt-injection condition.

In [ ]:
%%script /content/robustrag-py311/bin/python
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import re
import string
import sys

import joblib
import spacy
from nltk.corpus import stopwords

RUNNER_DIR = Path(os.environ["ROBUSTRAG_RUNNER_DIR"])
DATA_DIR = Path(os.environ["ROBUSTRAG_DATA_DIR"])
RUNS_DIR = Path(os.environ["ROBUSTRAG_RUNS_DIR"])
DATASETS = {"realtimeqa": ("realtimeqa.json", None), "open_nq": ("open_nq.json", None)}
if str(RUNNER_DIR) not in sys.path:
    sys.path.insert(0, str(RUNNER_DIR))
from src.dataset_utils import load_data
from src.models import BaseModel
from src.prompt_template import MISTRAL_TMPL

NLP = spacy.load("en_core_web_sm")
STOPWORDS = set(stopwords.words("english"))
IGNORED_POS = {"VERB", "INTJ", "ADP", "AUX", "CCONJ", "DET", "PART", "PRON", "SCONJ", "PUNCT", "SPACE"}
SEMANTIC_ABSTENTION = re.compile(r"\b(unknown|cannot determine|can't determine|insufficient information|not enough information|no information|unclear|unavailable)\b", re.I)

def exact_abstention(response):
    """Match the authors' literal abstention test exactly."""
    return "I don't" in response

def extract_keyword_set(response):
    """Mirror the upstream short-answer keyword extractor."""
    if exact_abstention(response):
        return set()
    document = NLP(response)
    phrases = [response.strip()]
    current = []
    for token in document:
        if token.pos_ in IGNORED_POS:
            if current:
                phrases.append("".join(item.lemma_ + item.whitespace_ for item in current).strip())
                phrases.extend(item.lemma_ for item in current)
                current = []
        else:
            current.append(token)
    phrases.append("".join(item.lemma_ + item.whitespace_ for item in current).strip())
    phrases.extend(item.lemma_ for item in current)
    return set(phrases)

def aggregate_keyword_sets(keyword_sets, alpha, beta):
    answering = [keywords for keywords in keyword_sets if keywords]
    threshold = min(beta, alpha * len(answering))
    counts = Counter(keyword for keywords in answering for keyword in keywords)
    retained = {
        keyword for keyword, count in counts.items()
        if count >= threshold and keyword not in string.punctuation and keyword not in STOPWORDS
    }
    return {"answering": len(answering), "threshold": threshold, "counts": counts, "retained": retained}

def normalized_text(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

def phrase_matches_gold(phrase, answer_aliases):
    phrase_normalized = normalized_text(phrase)
    if not phrase_normalized:
        return False
    for alias in answer_aliases:
        alias_normalized = normalized_text(alias)
        if not alias_normalized:
            continue
        if phrase_normalized == alias_normalized:
            return True
        if len(phrase_normalized) >= 3 and phrase_normalized in alias_normalized:
            return True
        if len(alias_normalized) >= 3 and alias_normalized in phrase_normalized:
            return True
    return False

def load_cached_isolated_responses(run_path, dataset):
    run_path = Path(run_path)
    cache_path = run_path / "cache" / f"mistral7b-{dataset}-10.z"
    if not cache_path.exists():
        raise FileNotFoundError(f"Missing response cache: {cache_path}")
    cache = joblib.load(cache_path)
    filename, _ = DATASETS[dataset]
    data_tool = load_data(dataset, 10, data_path=str(DATA_DIR / filename))
    sample_limit = json.loads((run_path / "manifest.json").read_text())["sample_limit"]
    prompt_builder = BaseModel()
    prompt_builder.prompt_template = MISTRAL_TMPL
    records = []
    for query_index, raw_item in enumerate(data_tool.data[:sample_limit]):
        item = data_tool.process_data_item(raw_item)
        prompts = prompt_builder.wrap_prompt(item, as_multi_choice=False, seperate=True)
        missing = [prompt for prompt in prompts if prompt not in cache]
        if missing:
            raise RuntimeError(f"Query {query_index} is missing {len(missing)} isolated responses from the cache.")
        responses = [cache[prompt] for prompt in prompts]
        records.append({"query_index": query_index, "item": item, "responses": responses})
    return data_tool, records

def load_clean_final_answers(run_path):
    result_files = sorted((Path(run_path) / "result").glob("*keyword*.json"))
    if not result_files:
        return {}
    payload = json.loads(result_files[-1].read_text())
    return {entry["query"]: entry["defended"] for entry in payload if "defended" in entry}

def profile_attack_candidates(run_path, dataset, alpha, beta, max_budget=3):
    data_tool, cached_records = load_cached_isolated_responses(run_path, dataset)
    clean_answers = load_clean_final_answers(run_path)
    findings = []
    profiles = []

    for cached in cached_records:
        item = cached["item"]
        responses = cached["responses"]
        answer_aliases = item["answer"]
        keyword_sets = [extract_keyword_set(response) for response in responses]
        clean = aggregate_keyword_sets(keyword_sets, alpha, beta)
        clean_gold = {word for word in clean["retained"] if phrase_matches_gold(word, answer_aliases)}
        final_answer = clean_answers.get(item["question"])
        clean_correct = None if final_answer is None else data_tool.eval_response(final_answer, item)
        gold_counts = {
            phrase: count for phrase, count in clean["counts"].items()
            if phrase_matches_gold(phrase, answer_aliases)
        }
        wrong_counts = {
            phrase: count for phrase, count in clean["counts"].items()
            if phrase and phrase not in string.punctuation and phrase not in STOPWORDS
            and not phrase_matches_gold(phrase, answer_aliases)
        }
        strongest_wrong = max(wrong_counts, key=lambda phrase: (wrong_counts[phrase], len(phrase), phrase), default=None)
        profiles.append({
            "query_index": cached["query_index"], "question": item["question"],
            "answers": answer_aliases, "clean_correct": clean_correct,
            "answering_passages": clean["answering"],
            "abstaining_passages": len(responses) - clean["answering"],
            "correct_keyword_max_count": max(gold_counts.values(), default=0),
            "gold_aligned_phrase_counts": gold_counts,
            "strongest_incorrect_phrase": strongest_wrong,
            "strongest_incorrect_count": 0 if strongest_wrong is None else wrong_counts[strongest_wrong],
            "clean_threshold": clean["threshold"],
        })

        distractors = {}
        for response in responses:
            if not response.strip() or exact_abstention(response) or SEMANTIC_ABSTENTION.search(response):
                continue
            for candidate in extract_keyword_set(response):
                if not candidate or candidate in string.punctuation or candidate in STOPWORDS:
                    continue
                if phrase_matches_gold(candidate, answer_aliases) or candidate in clean["retained"]:
                    continue
                distractors.setdefault(candidate, response.strip())

        for budget in range(1, max_budget + 1):
            survivors = keyword_sets[:-budget]

            for target, source_response in distractors.items():
                promoted = aggregate_keyword_sets(
                    survivors + [extract_keyword_set(target) for _ in range(budget)],
                    alpha,
                    beta,
                )
                if target in promoted["retained"]:
                    findings.append({
                        "attack": "promotion", "query_index": cached["query_index"],
                        "question": item["question"], "answers": answer_aliases,
                        "clean_correct": clean_correct, "budget": budget, "target": target,
                        "source_response": source_response,
                        "clean_target_count": clean["counts"][target],
                        "clean_threshold": clean["threshold"], "attacked_threshold": promoted["threshold"],
                    })

            filler_sets = [extract_keyword_set(f"Unknown marker {index + 1}") for index in range(budget)]
            survivor_state = aggregate_keyword_sets(survivors, alpha, beta)
            suppressed = aggregate_keyword_sets(survivors + filler_sets, alpha, beta)
            survivor_gold = {word for word in survivor_state["retained"] if phrase_matches_gold(word, answer_aliases)}
            attacked_gold = {word for word in suppressed["retained"] if phrase_matches_gold(word, answer_aliases)}
            if clean_gold and not attacked_gold:
                mechanism = "vote_loss" if not survivor_gold else "threshold_suppression"
                findings.append({
                    "attack": mechanism, "query_index": cached["query_index"],
                    "question": item["question"], "answers": answer_aliases,
                    "clean_correct": clean_correct, "budget": budget,
                    "clean_gold_keywords": sorted(clean_gold),
                    "surviving_gold_keywords": sorted(survivor_gold),
                    "clean_answering": clean["answering"], "attacked_answering": suppressed["answering"],
                    "clean_threshold": clean["threshold"], "survivor_threshold": survivor_state["threshold"],
                    "attacked_threshold": suppressed["threshold"],
                })

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_dir = RUNS_DIR / f"attack-profile-{dataset}-alpha{alpha}-{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "findings.json").write_text(json.dumps(findings, indent=2))
    for profile in profiles:
        profile["labels_by_budget"] = {}
        for budget in range(1, max_budget + 1):
            mechanisms = {
                row["attack"] for row in findings
                if row["query_index"] == profile["query_index"] and row["budget"] == budget
            }
            vulnerable = {name for name in mechanisms if name in {"promotion", "threshold_suppression"}}
            if vulnerable == {"promotion", "threshold_suppression"}:
                label = "both"
            elif vulnerable:
                label = next(iter(vulnerable))
            else:
                label = "safe"
            profile["labels_by_budget"][str(budget)] = {"label": label, "mechanisms": sorted(mechanisms)}
    (output_dir / "profiles.json").write_text(json.dumps(profiles, indent=2))
    summary = {}
    for attack in ["promotion", "threshold_suppression", "vote_loss"]:
        matching = [row for row in findings if row["attack"] == attack]
        summary[attack] = {
            "candidate_rows": len(matching),
            "unique_queries": len({row["query_index"] for row in matching}),
            "clean_correct_queries": len({row["query_index"] for row in matching if row["clean_correct"] is True}),
        }
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    print("Saved:", output_dir)
    return output_dir, findings

def review_candidates(findings, attack, limit=15):
    seen = set()
    shown = 0
    for row in findings:
        if row["attack"] != attack or row["query_index"] in seen:
            continue
        seen.add(row["query_index"])
        print("\nQUESTION:", row["question"])
        print("GOLD:", row["answers"])
        print("BUDGET:", row["budget"])
        if "target" in row:
            print("PROMOTION TARGET:", row["target"])
        else:
            print("CLEAN GOLD KEYWORDS:", row["clean_gold_keywords"])
            print("THRESHOLDS:", row["clean_threshold"], "->", row["attacked_threshold"])
        shown += 1
        if shown >= limit:
            break

candidate_runs = []
for candidate in RUNS_DIR.glob("table1-current_v2-realtimeqa-*"):
    manifest_path = candidate / "manifest.json"
    if manifest_path.exists() and json.loads(manifest_path.read_text()).get("status") == "complete":
        candidate_runs.append(candidate)
if not candidate_runs:
    raise RuntimeError("Complete the current_v2 RealtimeQA run before profiling attacks.")
current_rqa_run = max(candidate_runs, key=lambda path: path.stat().st_mtime_ns)
for alpha in (0.2, 0.3):
    rqa_attack_dir, rqa_findings = profile_attack_candidates(
        run_path=current_rqa_run, dataset="realtimeqa", alpha=alpha, beta=3.0, max_budget=3,
    )
    for attack_name in ("promotion", "threshold_suppression", "vote_loss"):
        print("\n=== alpha", alpha, attack_name, "===")
        review_candidates(rqa_findings, attack_name, limit=15)

### Profile RealtimeQA first

Run this on the completed current-paper cache. Do not construct live attacks until the printed examples have been manually checked.

In [ ]:
rqa_attack_dir = max(
    RUNS_DIR.glob("attack-profile-realtimeqa-alpha0.2-*"),
    key=lambda path: path.stat().st_mtime_ns,
)
rqa_findings = json.loads((rqa_attack_dir / "findings.json").read_text())
print((rqa_attack_dir / "summary.json").read_text())

In [ ]:
for attack_name in ("promotion", "threshold_suppression", "vote_loss"):
    examples = {}
    for row in rqa_findings:
        if row["attack"] == attack_name:
            examples.setdefault(row["query_index"], row)
    print(f"\n=== {attack_name}: {len(examples)} unique queries ===")
    for row in list(examples.values())[:15]:
        print("\nQUESTION:", row["question"])
        print("GOLD:", row["answers"])
        print("BUDGET:", row["budget"])
        print("TARGET:", row.get("target", row.get("clean_gold_keywords")))

## Finish

After confirming that all files have finished writing, unmount Drive. Then run `./pull.sh` locally to download the shared outputs. Before committing this notebook, use **Edit → Clear all outputs** so model logs and environment details are not stored in git.

In [ ]:
drive.flush_and_unmount()